In [141]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
## 데이터셋, 데이터로더 관련 모듈
from torch.nn.utils.rnn import pad_sequence
## 데이터 길이 맞추기

## 토커나이저, 단어사전 관련 모듈
from torchtext.data.utils import get_tokenizer              ## 토커나이저 인스턴스 추출
from torchtext.vocab import build_vocab_from_iterator       ## 데이터셋에서 단어사전 생성 함수
from nltk.corpus import stopwords                           ## 불용어 데이터셋
import nltk
from nltk.tokenize import word_tokenize

In [142]:
dataDF = pd.read_csv('./dataDF.csv', index_col=0, encoding='utf-8')

In [143]:
texts = dataDF['Dialogue'].tolist()
labels = dataDF['age'].tolist()

In [144]:
### ===> 모듈 로딩
from konlpy.tag import Okt
from torchtext.vocab import build_vocab_from_iterator
import string
### ===> 토큰관련 특별 문자
UNK = '<UNK>'
PAD = '<PAD>'
### 토큰화 인스턴스 생성
tokenizer = Okt()
### ===> 토큰 제너레이터 함수 : 데이터 추출하여 토큰화 


In [145]:
PUNC = string.punctuation
STOPWORDS = [
    # 조사 / 불필요 접속어
    '은', '는', '이', '가', '을', '를', '에', '에서', '으로', '의', '도', '만', '까지', '부터',
    '과', '와', '하고', '보다', '보다도',

    # 보조동사 및 흔한 표현
    '있다', '없다', '되다', '해요', '해', '했어요', '했네', '했지', '하지', '그랬지', '그래요', '같아요',
    
    # 대명사 / 불분명 주어
    '그', '저', '이', '것', '거', '자기', '우리', '너', '나', '누구', '사람', '다', '뭐',

    # 웹체 / 감탄 / 의미 낮은 부사
    '정말', '그냥', '좀', '매우', '많이', '아주', '거의', '조금', '계속', '항상', '진짜',
    
    # 대화 문법 전환
    '그래서', '그런데', '그러니까', '하지만', '그리고', '그러면',

    # 의성어 / 감탄사 / 불필요한 감정어
    '티티', '아가씨', '수고했다', '감사합니다', '고맙다', '고마워서', '선물', '미신이야',

    # 웹 말투
    'ㅋㅋ', 'ㅎㅎ', 'ㅠㅠ', '...', '!!', '??'
]
UNK, PAD  = '<UNK>',  '<PAD>'

In [146]:
def yield_tokens(data):
    for line in data:
        line = ''.join([x for x in line if x not in PUNC])
        yield word_tokenize(line.lower())


In [147]:
VOCAB = build_vocab_from_iterator(yield_tokens(texts), specials=[UNK, PAD])
VOCAB.set_default_index(VOCAB[UNK])

In [148]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in texts]

In [149]:
from gensim.models import Word2Vec


word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=10,
    max_final_vocab=10000
)

# 2. UNK 토큰 추가
unk_vector = np.random.normal(scale=0.6, size=(word2vec.vector_size,))
word2vec.wv.add_vector("<UNK>", unk_vector)

# 3. 단어 인덱스 매핑
word2index = word2vec.wv.key_to_index
unk_index = word2index["<UNK>"]

# 4. 임베딩 weight → PyTorch 임베딩 레이어
embedding_weights = torch.FloatTensor(word2vec.wv.vectors)
embedding_layer = torch.nn.Embedding.from_pretrained(embedding_weights, freeze=False)

c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\gensim\models\keyedvectors.py:551: UserWarning: Adding single vectors to a KeyedVectors which grows by one each time can be costly. Consider adding in batches or preallocating to the required size.
  warnings.warn(


In [150]:
## 배치크기만큼 데이터 로딩 시 위치 지정 위한 설정 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [151]:
# 레이블 매핑 딕셔너리 정의
label_map = {'80': 0, '90': 1, '00': 2, '10': 3}
# 예상치 못한 레이블이 들어올 경우 처리할 기본값 (예: 0 또는 특정 UNK 레이블 인덱스)
default_label_index = 0 # 또는 -1 등으로 설정 후 후처리

In [152]:
def label_pipeline(label_str):
    """주어진 문자열 레이블('80', '90', '00', '10')을 정수 인덱스(0, 1, 2, 3)로 변환"""
    index = label_map.get(label_str, default_label_index)
    if label_str not in label_map:
         print(f"Warning: 예상치 못한 레이블 '{label_str}' 발견. 기본값 {default_label_index}로 처리합니다.")
    return index

# 텍스트 파이프라인 (이전 코드와 동일하게 유지)
def text_pipeline(text):
    tokens = tokenizer.morphs(text) # Okt 토크나이저 사용
    # word2index 딕셔너리를 사용하여 인덱스로 변환, 없으면 unk_index 사용
    return [word2index.get(token, unk_index) for token in tokens]

In [153]:
## ----------------------------------------------------------------------
## 함수기능 : 배치크기 만큼 데이터셋 로딩해서 토큰 + 텐서화 진행 후 반환
## ----------------------------------------------------------------------
def collate_batch(batch):
    ## 라벨, 뉴스, 뉴스기사 시작 위치값 저장 변수
    diag_list, label_list, offsets = [], [], [0]

    ## 1개씩 라벨과 뉴스 기사 추출
    for news, label in batch:
        ## 라벨 인코딩 후 추가 : 1 ~ 4 => 0 ~ 3
        label_list.append(label_pipeline(label))

        ## 뉴스 기사 인코딩 후 추가 
        processed_news = torch.tensor(text_pipeline(news), dtype=torch.int64)
        diag_list.append(processed_news)

        ## 다음 뉴스를 읽기 위한 위치값 정보
        offsets.append(processed_news.size(0))
        #print(f'news 토큰 수 => {processed_news.size(0)}개')

    ## 배치 크기 만큼의 라벨 리스트 => 텐서화
    label_list = torch.tensor(label_list, dtype=torch.int64)

    ## 배치 크기 만큼의 길이 위치값 => 텐서화 
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    ## 배치 크기 만큼의 뉴스 기사 리스트 => 텐서화 
    diag_list = torch.cat(diag_list)

    return label_list.to(DEVICE), diag_list.to(DEVICE), offsets.to(DEVICE)


In [154]:
class customDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

train_dataset = customDataset(X_train, y_train)
val_dataset = customDataset(X_val, y_val)
test_dataset = customDataset(X_test, y_test)

# train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)
# test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

In [156]:
train_dataset[0]

('이거 안 받으실 거예요?', 0)

In [157]:
## => DataLoader 생성
### ===> 학습용, 검증용, 테스트용 DataSet 준비 
BATCH_SIZE = 1000

### 학습용, 검증용, 테스트용 Dataset, DataLoader 준비
trainDL = DataLoader( train_dataset, 
                      batch_size=BATCH_SIZE, 
                      shuffle=True, 
                      collate_fn=collate_batch )

validDL  = DataLoader( val_dataset, 
                       batch_size=BATCH_SIZE,  
                       shuffle=True,  
                       collate_fn=collate_batch )
                      
testDL  = DataLoader( test_dataset, 
                      batch_size=BATCH_SIZE,  
                      shuffle=True,  
                      collate_fn=collate_batch )

In [158]:
for idx, (label, text) in enumerate(trainDL):
    print( idx, label.shape, text.shape)
    break

ValueError: too many values to unpack (expected 2)

In [ ]:
### ===> 모듈로딩
import torch.nn as nn
import torch.optim as optim 
from torch.optim.lr_scheduler import StepLR

In [ ]:
## -------------------------------------------------------------------------
## 클래스이름 : TextModel
## 부모클래스 : Module
## 매개변수둘 : 단어사전 갯수, 임베딩 수, 2진분류
## -------------------------------------------------------------------------
class TextDnnEmbModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_dim=4):
        super().__init__()
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 4)  # 4개 클래스
        )

    def forward(self, text):
        embedded = self.embedding_layer(text)  # (batch_size, seq_len, emb_dim)
        pooled = embedded.mean(dim=1)          # (batch_size, emb_dim)
        return self.classifier(pooled)


    

In [ ]:
## 학습 설정
INPUT_SIZE      = 256
LR              = 0.01
EPOCHS          = 10
STEP_SIZE       = 5
NUM_CLASS       = 1

EMBEDDING_DIM   = 128
HIDDEN_DIM      = 128
VOCAB_SIZE      = len(VOCAB)
# VOCAB_SIZE      = len(word2vec.wv.key_to_index)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
## 학습 인스턴스 생성
MODEL = TextDnnEmbModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
MODEL.to(DEVICE)

LOSS_FN   = nn.CrossEntropyLoss()
OPTIMIZER = optim.Adam(MODEL.parameters(), lr=LR)
SCHEDULER = StepLR(OPTIMIZER, STEP_SIZE, gamma=0.1)
""" 
Decays the learning rate of each parameter group by gamma every step_size epochs. 
Notice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. 
When last_epoch=-1, sets initial lr as lr.

"""
## 전에 쓴 건 스코어가 변하지 않으면 patience만큼 기다렸다가 학습중지


' \nDecays the learning rate of each parameter group by gamma every step_size epochs. \nNotice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. \nWhen last_epoch=-1, sets initial lr as lr.\n\n'

In [ ]:
for idx, (label, text) in enumerate(trainDL):
    print( idx, label.shape, text.shape)
    break

ValueError: too many values to unpack (expected 2)

In [ ]:
def training(dataloader):
## -------------------------------------------------------------------------
## 함수기능 : 학습데이터를 사용하여 학습 진행
## 함수이름 : training
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------

    ## 학습 모드 설정
    MODEL.train()

    ## 학습 손실과 점수 저장 
    total_loss, total_acc = 0, 0
    
    for idx, (text, label) in enumerate(dataloader):

        OPTIMIZER.zero_grad()
        pre  = MODEL(text)

        loss = LOSS_FN(pre, label.reshape(-1).long())
        loss.backward()
        ## gradient vanishing, gradient exploding 발생 => 방지 및 안정화 
        ## - gradient가 일정 threshold를 넘어가면 clipping
        ## - clipping: gradient의 L2norm(norm이지만 보통 L2 norm사용)으로 나눠주는 방식
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 0.1)
        OPTIMIZER.step()

        total_loss += loss.item()
        total_acc += (pre.argmax(dim=1) == label).sum().item()

        if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1



In [ ]:
def evaluate(dataloader):
## -------------------------------------------------------------------------
## 함수기능 : 검증데이터를 사용하여 학습 진행
## 함수이름 : evaluate
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------

    MODEL.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for idx, (text, label) in enumerate(dataloader):
            ## 추론 진행
            pre = MODEL(text)
            ## 손실 계산
            loss = LOSS_FN(pre, label.reshape(-1).long())

            ## 손실 및 성능 평가
            total_loss += loss.item()
            total_acc += (pre.argmax(1) == label).sum().item()
            if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1

In [ ]:
## 모델 및 모델 층별 상태값 즉, 파라미터 값 저장 경로
MODEL_DIR  = './models/'
MODEL_FILE = 'IMDB_DNN_MODEL.pt'


In [ ]:
# EPOCHS = 100  ## 임시
# # 모델 저장 기준
# MAX_ACC = 0.

# for epoch in range(1, EPOCHS + 1):
    
#     train_loss, train_acc = training(train_loader)
#     valid_loss, valid_acc = evaluate(val_loader)
#     SCHEDULER.step()

#     print("-" * 59)
#     print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
#     print("-" * 59)

#     ## 모델 저장 
#     if MAX_ACC < valid_acc : 
#         torch.save(MODEL, MODEL_DIR+MODEL_FILE)
#         MAX_ACC = valid_acc


In [ ]:
import os

EPOCHS = 100
PATIENCE = 25
patience_counter = 0
MAX_ACC = 0.

for epoch in range(1, EPOCHS + 1):
    
    train_loss, train_acc = training(trainDL)
    valid_loss, valid_acc = evaluate(validDL)
    SCHEDULER.step()

    print("-" * 59)
    print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
    print("-" * 59)

    # 모델 저장 디렉토리 없으면 생성
    os.makedirs(MODEL_DIR, exist_ok=True)

    # 성능 향상 시 저장 (epoch 번호 포함)
    if valid_acc > MAX_ACC and valid_acc > 60:
        model_path = os.path.join(MODEL_DIR, f"epoch{epoch}_v{valid_acc:.2f}.pt")
        torch.save(MODEL, model_path)
        MAX_ACC = valid_acc
        patience_counter = 0
        print(f"✔ 모델 저장됨: {model_path}")
    else:
        patience_counter += 1
        print(f'→ No improvement. Patience counter: {patience_counter}/{PATIENCE}')

        if patience_counter >= PATIENCE:
            print(f'→ Early stopping at epoch {epoch} (best val acc: {MAX_ACC:.3f})')
            break


ValueError: too many values to unpack (expected 2)